# F11 - Neural network fundamentals

**Hardware:** CPU-only · **Time:** 3 hours · **Prerequisite:** F10

A neural network is a stack of differentiable functions. This notebook starts
with a single perceptron, shows why one linear boundary cannot solve XOR, builds
a tiny multilayer network by hand, and then trains a scikit-learn multilayer
perceptron on a nonlinear classification task.

In [1]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

SEED = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
print("numpy", np.__version__)

numpy 2.5.4


## 1. A perceptron is a weighted sum plus a step

The perceptron computes `step(w1*x1 + w2*x2 + b)`. With the weights below it
implements logical OR. The same structure cannot implement XOR, because XOR is
not linearly separable.

In [2]:
def step(value):
    return (value >= 0).astype(int)


def perceptron(x1, x2, w1=1.0, w2=1.0, bias=-0.5):
    return step(w1 * x1 + w2 * x2 + bias)


inputs = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
or_output = perceptron(inputs[:, 0], inputs[:, 1])
print("OR:", list(zip(map(tuple, inputs), or_output)))
assert or_output.tolist() == [0, 1, 1, 1]

xor_target = np.array([0, 1, 1, 0])
xor_output = perceptron(inputs[:, 0], inputs[:, 1], bias=-0.5)
print("single perceptron on XOR:", xor_output.tolist(), "target:", xor_target.tolist())
assert xor_output.tolist() != xor_target.tolist()

OR: [((np.int64(0), np.int64(0)), np.int64(0)), ((np.int64(0), np.int64(1)), np.int64(1)), ((np.int64(1), np.int64(0)), np.int64(1)), ((np.int64(1), np.int64(1)), np.int64(1))]
single perceptron on XOR: [0, 1, 1, 1] target: [0, 1, 1, 0]


## 2. A hidden layer changes the geometry

Two hidden units can represent OR and AND. The output unit combines them as
`OR - AND`, which is XOR. This is a hand-built example, not a trained model, but
it shows what a hidden layer buys you: a new representation in which the problem
is linearly separable.

In [3]:
def two_layer_xor(x1, x2):
    hidden_or = step(x1 + x2 - 0.5)
    hidden_and = step(x1 + x2 - 1.5)
    return step(hidden_or - hidden_and - 0.5)


xor_output = two_layer_xor(inputs[:, 0], inputs[:, 1])
print("two-layer XOR:", xor_output.tolist())
assert xor_output.tolist() == xor_target.tolist()

two-layer XOR: [0, 1, 1, 0]


## 3. Activation functions

Without a nonlinear activation, stacking layers is still a linear function.
ReLU is cheap and common; sigmoid squashes to (0, 1); tanh squashes to (-1, 1).
ReLU can produce dead units when a unit receives only negative inputs.

In [4]:
z = np.linspace(-5, 5, 200)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(z, np.maximum(0, z), label="ReLU")
ax.plot(z, 1 / (1 + np.exp(-z)), label="sigmoid")
ax.plot(z, np.tanh(z), label="tanh")
ax.set_xlabel("input")
ax.set_ylabel("activation")
ax.set_title("Common activation functions")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "f11_activations.png", bbox_inches="tight")
plt.close(fig)
assert (OUTPUT_DIR / "f11_activations.png").exists()

## 4. A nonlinear classification problem

Two moons cannot be separated by a straight line. Logistic regression finds a
linear boundary; a multilayer perceptron learns a curved one. Compare both on the
same split and inspect the decision boundary.

In [5]:
X, y = make_moons(n_samples=600, noise=0.25, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

logistic = LogisticRegression(max_iter=2000, random_state=SEED).fit(X_train_s, y_train)
mlp = MLPClassifier(
    hidden_layer_sizes=(16, 8),
    activation="relu",
    solver="adam",
    max_iter=2000,
    random_state=SEED,
).fit(X_train_s, y_train)

for name, model in {"logistic": logistic, "mlp": mlp}.items():
    print(name, "test accuracy:", round(accuracy_score(y_test, model.predict(X_test_s)), 3))
assert accuracy_score(y_test, mlp.predict(X_test_s)) > accuracy_score(y_test, logistic.predict(X_test_s))

logistic test accuracy: 0.86
mlp test accuracy: 0.94


## 5. Inspect the learned shapes

The first layer maps two input features to 16 hidden units. The second maps 16
to 8. The output layer produces one logit. The number of parameters is the sum of
weights plus biases; it is not the same as the number of neurons.

In [6]:
for index, (weights, bias) in enumerate(zip(mlp.coefs_, mlp.intercepts_), start=1):
    print(f"layer {index}: weights {weights.shape}, bias {bias.shape}")
parameter_count = sum(weights.size + bias.size for weights, bias in zip(mlp.coefs_, mlp.intercepts_))
print("trainable parameters:", parameter_count)
assert mlp.coefs_[0].shape == (2, 16)

layer 1: weights (2, 16), bias (16,)
layer 2: weights (16, 8), bias (8,)
layer 3: weights (8, 1), bias (1,)
trainable parameters: 193


## 6. Visualize the decision boundary

The grid below evaluates the model at many points. The curved region is the
network's learned representation of "moon or not moon". A flexible boundary can
also fit noise, which is why validation data and regularization matter.

In [7]:
xx, yy = np.meshgrid(np.linspace(-2.5, 3.5, 250), np.linspace(-2.0, 2.5, 250))
grid = scaler.transform(np.c_[xx.ravel(), yy.ravel()])
zz = mlp.predict(grid).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.contourf(xx, yy, zz, alpha=0.2, cmap="coolwarm")
ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="coolwarm", edgecolor="k", s=18)
ax.set_xlabel("feature 1")
ax.set_ylabel("feature 2")
ax.set_title("MLP decision boundary on two moons")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "f11_decision_boundary.png", bbox_inches="tight")
plt.close(fig)
assert (OUTPUT_DIR / "f11_decision_boundary.png").exists()

## Check your work

- [ ] You can explain why a single perceptron cannot solve XOR.
- [ ] You can name the role of a hidden layer and a nonlinear activation.
- [ ] You can read the parameter shapes of an MLP.
- [ ] You can explain why a curved boundary can overfit.
- [ ] `outputs/f11_activations.png` and `outputs/f11_decision_boundary.png` exist.

**Next:** F12 - Tensors, gradients, loss functions, optimizers, and backpropagation.